# HMC Run Analysis — wclover_2p1
Equilibration, time series, autocorrelation, and dH diagnostics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hmc import load_run
from hmc.autocorr import gamma_method
from hmc.equilibrate import suggest_burnin

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

BASE = '/lustre2/nplqcd/vayyar/grid_qcd/runs'

In [ ]:
df = load_run(f'{BASE}/wclover_2p1/hmc')
print(f"Loaded {len(df)} trajectories  (traj {df.traj.min()} – {df.traj.max()})")
if df.accepted.notna().any():
    print(f"Acceptance rate: {df.accepted.mean():.1%}")
df.head(10)

## Equilibration

In [ ]:
burnin = suggest_burnin(df, observable='plaquette')
print(f"Suggested burn-in: traj {burnin}")

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for ax, col, label in zip(axes,
                          ['plaquette', 'polyakov_abs'],
                          ['Plaquette', '|Polyakov loop|']):
    ax.plot(df.traj, df[col], 'o-', ms=3, lw=0.8)
    if burnin is not None:
        ax.axvline(burnin, color='red', ls='--', lw=1.5, label=f'burn-in (traj {burnin})')
        ax.legend(fontsize=9)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Trajectory')
fig.suptitle('Time series — wclover_2p1', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Apply burn-in cut — adjust manually if needed
burnin_cut = burnin if burnin is not None else df.traj.min()
prod = df[df.traj >= burnin_cut].copy()
print(f"Production sample: {len(prod)} trajectories (traj {prod.traj.min()} – {prod.traj.max()})")

## Autocorrelation

In [ ]:
observables = {'plaquette': 'Plaquette', 'polyakov_abs': '|Polyakov loop|', 'dH': 'dH'}

results = {}
for col, label in observables.items():
    data = prod[col].dropna().values
    if len(data) < 4:
        continue
    try:
        res = gamma_method(data)
        results[label] = res
    except Exception as e:
        print(f"{label}: {e}")

fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4))
if len(results) == 1:
    axes = [axes]
for ax, (label, res) in zip(axes, results.items()):
    lags = np.arange(len(res['rho']))
    ax.bar(lags, res['rho'], width=0.8, color='steelblue', alpha=0.7)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xlabel('Lag')
    ax.set_ylabel('ρ(t)')
    ax.set_title(f'{label}\nτ_int = {res["tau_int"]:.2f} ± {res["tau_int_err"]:.2f}')
    ax.grid(True, alpha=0.3)
plt.suptitle('Autocorrelation functions (Gamma method)', fontsize=12)
plt.tight_layout()
plt.show()

## dH diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

dH = prod['dH'].dropna()
axes[0].hist(dH, bins=20, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(0, color='red', ls='--', lw=1.2)
axes[0].set_xlabel('dH')
axes[0].set_ylabel('Count')
axes[0].set_title(f'dH  (mean = {dH.mean():.4f})')
axes[0].grid(True, alpha=0.3)

exp_dH = prod['exp_dH'].dropna()
axes[1].hist(exp_dH, bins=20, color='coral', alpha=0.7, edgecolor='white')
axes[1].axvline(1, color='red', ls='--', lw=1.2, label='ideal = 1')
axes[1].set_xlabel('exp(−dH)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'exp(−dH)  (mean = {exp_dH.mean():.4f})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Integrator quality', fontsize=13)
plt.tight_layout()
plt.show()

## Summary

In [ ]:
rows = []
for label, res in results.items():
    n = len(prod[next(k for k, v in observables.items() if v == label)].dropna())
    rows.append({
        'Observable' : label,
        'Mean'       : f"{res['mean']:.6f}",
        'Stat. error': f"{res['sigma']:.6f}",
        'τ_int'      : f"{res['tau_int']:.2f} ± {res['tau_int_err']:.2f}",
        'Window'     : res['window'],
        'N_eff'      : f"{n / (2 * res['tau_int']):.1f}",
    })
pd.DataFrame(rows).set_index('Observable')